# 🎓 Predict Success: Build ML Models for Graduation & Placement Forecasting

---

**Project Type:** Internship / Academic Project  
**Author:** Student  
**Date:** June 2026  
**Python Version:** 3.x  

---

## 1. Problem Statement

### 1.1 Business Problem

Educational institutions face significant challenges in identifying students who are at risk of **not graduating** or **not getting placed** after completing their studies. Early identification of such students allows institutions to intervene proactively — through counseling, academic support, or placement training programs — to improve outcomes.

### 1.2 Objectives

This project aims to:

1. **Predict Graduation Status:** Build a binary classification model to predict whether a student will graduate successfully (`1`) or not (`0`).
2. **Predict Placement Status:** Build a binary classification model to predict whether a student will get placed (`1`) or not (`0`).
3. **Identify Key Factors:** Determine which student attributes (CGPA, attendance, skills, etc.) have the most influence on graduation and placement outcomes.
4. **Provide Actionable Recommendations:** Offer data-driven insights that institutions can use to improve student success rates.

### 1.3 How Machine Learning Can Help

Machine learning models can analyze historical student data and learn complex, non-linear patterns that traditional statistical methods might miss. Specifically:

- **Early Warning Systems:** ML models can flag at-risk students early in their academic journey, enabling timely intervention.
- **Resource Optimization:** Institutions can allocate resources (tutoring, mentoring, placement training) more efficiently by targeting students who need them most.
- **Data-Driven Decision Making:** Instead of relying on intuition, administrators can use model predictions and feature importance analysis to design better curricula and support programs.
- **Scalability:** Once trained, these models can process thousands of student records in seconds, making them suitable for large institutions.

---

## 2. Import Libraries

We begin by importing all the necessary Python libraries for data manipulation, visualization, machine learning modeling, and evaluation. Each library serves a specific purpose:

- **pandas & numpy:** Data manipulation and numerical operations.
- **matplotlib & seaborn:** Data visualization and plotting.
- **scikit-learn:** Machine learning algorithms, preprocessing, and evaluation metrics.
- **xgboost:** Advanced gradient boosting algorithm (optional but recommended).

In [ ]:
# ============================================================
# 2. Import Libraries
# ============================================================

# --- Core Data Libraries ---
import pandas as pd
import numpy as np
import os

# --- Visualization Libraries ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Preprocessing ---
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler

# --- Machine Learning Models ---
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# --- XGBoost (optional — gracefully handled if not installed) ---
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print("✅ XGBoost is available and imported successfully.")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️ XGBoost is not installed. Skipping XGBoost models.")
    print("   Install with: pip install xgboost")

# --- Evaluation Metrics ---
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

# --- Suppress Warnings for Cleaner Output ---
import warnings
warnings.filterwarnings('ignore')

# --- Plot Settings ---
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print("\n✅ All libraries imported successfully!")
print(f"   pandas: {pd.__version__}")
print(f"   numpy: {np.__version__}")

## 3. Data Loading

The dataset is stored in the `data/` folder as `student_data.csv`. This CSV was generated by the standalone script `data/generate_dataset.py`, which creates a realistic synthetic student dataset with:

- **1,005 student records** (1,000 unique + 5 intentional duplicates for preprocessing practice)
- **15 columns** (13 features + 2 target variables)
- **~2% missing values** in selected columns (Attendance, Internal Marks, Communication Skills, Aptitude Score)

### Dataset Schema

| Column | Type | Description |
|--------|------|-------------|
| Student_ID | object | Unique student identifier |
| Gender | object | Male / Female |
| Age | int | Student age (18–26) |
| Attendance_Percentage | float | Class attendance (20–100%) |
| CGPA | float | Cumulative GPA (2.0–10.0) |
| Internal_Marks | float | Internal assessment score (10–100) |
| External_Marks | float | External exam score (5–100) |
| Backlogs | int | Number of backlogs (0–5) |
| Study_Hours_Per_Week | float | Weekly study hours (1–40) |
| Internship_Experience | int | Has internship? (0/1) |
| Projects_Completed | int | Number of projects (0–5) |
| Communication_Skills_Score | float | Communication rating (1–10) |
| Aptitude_Test_Score | float | Aptitude test score (10–100) |
| **Graduation_Status** | **int** | **Target 1 — Graduated? (0/1)** |
| **Placement_Status** | **int** | **Target 2 — Placed? (0/1)** |

> **To use your own dataset:** Replace `data/student_data.csv` with your CSV file. The code is written to adapt automatically.

In [ ]:
# ============================================================
# 3. Data Loading
# ============================================================

# --- Define path to dataset ---
DATA_DIR = os.path.join(os.getcwd(), 'data')
DATA_FILE = os.path.join(DATA_DIR, 'student_data.csv')

# --- Load dataset ---
try:
    df = pd.read_csv(DATA_FILE)
    print(f"✅ Dataset loaded successfully from: {DATA_FILE}")
except FileNotFoundError:
    raise FileNotFoundError(
        f"❌ Dataset not found at '{DATA_FILE}'.\n"
        f"   Please run 'python data/generate_dataset.py' first, "
        f"or place your own CSV at that path."
    )

print(f"\n📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# Display column names and data types
print("📋 Column Names and Data Types:")
print("=" * 50)
print(df.dtypes)
print(f"\n📐 Dataset Dimensions: {df.shape}")

In [ ]:
# Display first 5 records
print("🔍 First 5 Records:")
df.head()

In [ ]:
# Display last 5 records
print("🔍 Last 5 Records:")
df.tail()

In [ ]:
# Random sample of 5 records for quick inspection
print("🎲 Random Sample (5 Records):")
df.sample(5, random_state=42)

---

## 4. Exploratory Data Analysis (EDA)

Exploratory Data Analysis is a critical step in any data science project. It helps us:

- **Understand the structure** and quality of the data.
- **Identify patterns**, trends, and anomalies.
- **Detect missing values** and outliers that need treatment.
- **Understand relationships** between features and target variables.
- **Inform feature engineering** and model selection decisions.

We will perform the following analyses:
1. Dataset overview
2. Missing values analysis
3. Duplicate records check
4. Statistical summary
5. Distribution plots
6. Boxplots (outlier detection)
7. Correlation heatmap
8. Target variable analysis
9. Feature relationships

### 4.1 Dataset Overview

In [ ]:
# ============================================================
# 4.1 Dataset Overview
# ============================================================

print("📊 DATASET OVERVIEW")
print("=" * 60)
print(f"Number of Records  : {df.shape[0]}")
print(f"Number of Features : {df.shape[1]}")
print(f"\nTarget Variables:")
print(f"  • Graduation_Status : {df['Graduation_Status'].nunique()} classes")
print(f"  • Placement_Status  : {df['Placement_Status'].nunique()} classes")
print(f"\nFeature Types:")
print(f"  • Numerical  : {df.select_dtypes(include=[np.number]).shape[1]}")
print(f"  • Categorical: {df.select_dtypes(include=['object']).shape[1]}")

print("\n" + "=" * 60)
print("\n📋 Detailed Info:")
df.info()

### 4.2 Missing Values Analysis

Missing values can distort model training and lead to biased predictions. We need to identify which columns have missing data, how much is missing, and decide on an appropriate imputation strategy.

In [ ]:
# ============================================================
# 4.2 Missing Values Analysis
# ============================================================

missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Percentage': missing_pct
}).sort_values(by='Missing_Count', ascending=False)

print("🔍 MISSING VALUES ANALYSIS")
print("=" * 50)
print(missing_df[missing_df['Missing_Count'] > 0])

if missing.sum() == 0:
    print("\n✅ No missing values found in the dataset!")
else:
    print(f"\n⚠️ Total missing values: {missing.sum()}")

# --- Visualize missing values ---
fig, ax = plt.subplots(figsize=(10, 5))
cols_with_missing = missing_df[missing_df['Missing_Count'] > 0]
if len(cols_with_missing) > 0:
    cols_with_missing['Missing_Percentage'].plot(
        kind='barh', color='#e74c3c', edgecolor='black', ax=ax
    )
    ax.set_xlabel('Missing Percentage (%)', fontsize=12)
    ax.set_title('Missing Values by Feature', fontsize=14, fontweight='bold')
    ax.bar_label(ax.containers[0], fmt='%.1f%%', padding=3)
    plt.tight_layout()
    plt.show()
else:
    print("No missing values to plot.")

### 4.3 Duplicate Records Check

Duplicate records can inflate model performance metrics and skew statistical summaries. We identify and remove exact duplicates.

In [ ]:
# ============================================================
# 4.3 Duplicate Records Check
# ============================================================

# Check for duplicates (excluding Student_ID since it's a unique identifier)
feature_cols = [c for c in df.columns if c != 'Student_ID']
n_duplicates = df.duplicated(subset=feature_cols, keep='first').sum()

print(f"🔍 DUPLICATE RECORDS CHECK")
print("=" * 50)
print(f"Total duplicate rows found: {n_duplicates}")

if n_duplicates > 0:
    print(f"\n⚠️ {n_duplicates} duplicate rows detected. Removing...")
    df = df.drop_duplicates(subset=feature_cols, keep='first').reset_index(drop=True)
    print(f"✅ Duplicates removed. New shape: {df.shape}")
else:
    print("✅ No duplicate records found.")

### 4.4 Statistical Summary

The statistical summary provides a quick snapshot of the central tendency, dispersion, and shape of each numerical feature's distribution. Key metrics include mean, standard deviation, min, max, and quartiles.

In [ ]:
# ============================================================
# 4.4 Statistical Summary
# ============================================================

print("📊 STATISTICAL SUMMARY — Numerical Features")
print("=" * 60)
df.describe().T.style.background_gradient(cmap='YlOrRd', axis=1)

In [ ]:
# Statistical summary for categorical features
print("📊 STATISTICAL SUMMARY — Categorical Features")
print("=" * 60)
df.describe(include='object').T

### 4.5 Distribution Plots

Distribution plots (histograms with KDE curves) help us understand the shape of each feature's distribution — whether it's normally distributed, skewed, or multi-modal. This is important for selecting appropriate preprocessing techniques.

In [ ]:
# ============================================================
# 4.5 Distribution Plots
# ============================================================

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Exclude target and binary variables for distribution plots
dist_cols = [c for c in numerical_cols if c not in [
    'Graduation_Status', 'Placement_Status', 'Internship_Experience'
]]

n_cols = 3
n_rows = (len(dist_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6',
          '#1abc9c', '#e67e22', '#34495e', '#16a085']

for idx, col in enumerate(dist_cols):
    ax = axes[idx]
    sns.histplot(
        data=df, x=col, kde=True, ax=ax,
        color=colors[idx % len(colors)], edgecolor='white', alpha=0.7
    )
    ax.set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean: {df[col].mean():.2f}')
    ax.legend(fontsize=9)

# Hide unused subplots
for idx in range(len(dist_cols), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('📊 Feature Distribution Plots', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.6 Boxplots (Outlier Detection)

Boxplots provide a visual summary of the data's distribution and help identify **outliers** — data points that fall significantly outside the typical range. Outliers can negatively affect model training, especially for distance-based algorithms.

In [ ]:
# ============================================================
# 4.6 Boxplots (Outlier Detection)
# ============================================================

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for idx, col in enumerate(dist_cols):
    ax = axes[idx]
    sns.boxplot(
        data=df, y=col, ax=ax,
        color=colors[idx % len(colors)],
        width=0.4, flierprops=dict(marker='o', markersize=5, markerfacecolor='red')
    )
    ax.set_title(f'Boxplot: {col}', fontsize=12, fontweight='bold')
    ax.set_ylabel('')

for idx in range(len(dist_cols), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('📊 Boxplots for Outlier Detection', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.7 Correlation Heatmap

The correlation heatmap reveals **linear relationships** between numerical features. Strong positive or negative correlations between features and target variables indicate predictive power. High correlations between features themselves may indicate **multicollinearity**, which can affect certain models.

In [ ]:
# ============================================================
# 4.7 Correlation Heatmap
# ============================================================

# Compute correlation matrix for numerical columns
corr_matrix = df[numerical_cols].corr()

# Create mask for upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlBu_r', center=0, linewidths=0.5,
    square=True, ax=ax,
    cbar_kws={'shrink': 0.8, 'label': 'Correlation Coefficient'}
)
ax.set_title('🔗 Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# --- Display correlations with target variables ---
print("\n📊 Correlations with Graduation_Status:")
print(corr_matrix['Graduation_Status'].drop(
    ['Graduation_Status', 'Placement_Status']
).sort_values(ascending=False).to_string())

print("\n📊 Correlations with Placement_Status:")
print(corr_matrix['Placement_Status'].drop(
    ['Graduation_Status', 'Placement_Status']
).sort_values(ascending=False).to_string())

### 4.8 Target Variable Analysis

Understanding the distribution of target variables is crucial for assessing **class imbalance**. Highly imbalanced datasets may require special techniques such as resampling, class weighting, or alternative evaluation metrics.

In [ ]:
# ============================================================
# 4.8 Target Variable Analysis
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graduation Status
grad_counts = df['Graduation_Status'].value_counts()
colors_target = ['#e74c3c', '#2ecc71']
labels = ['Not Graduated (0)', 'Graduated (1)']

axes[0].pie(
    grad_counts, labels=labels, colors=colors_target,
    autopct='%1.1f%%', startangle=90, explode=(0.05, 0),
    shadow=True, textprops={'fontsize': 11}
)
axes[0].set_title('🎓 Graduation Status Distribution', fontsize=14, fontweight='bold')

# Placement Status
place_counts = df['Placement_Status'].value_counts()
labels_p = ['Not Placed (0)', 'Placed (1)']
colors_p = ['#e67e22', '#3498db']

axes[1].pie(
    place_counts, labels=labels_p, colors=colors_p,
    autopct='%1.1f%%', startangle=90, explode=(0.05, 0),
    shadow=True, textprops={'fontsize': 11}
)
axes[1].set_title('💼 Placement Status Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n📊 Graduation Status Counts:\n{grad_counts.to_string()}")
print(f"\n📊 Placement Status Counts:\n{place_counts.to_string()}")

### 4.9 Feature Relationships with Target Variables

Visualizing how individual features relate to the target variables helps us understand which features are most discriminative and may inform feature selection.

In [ ]:
# ============================================================
# 4.9a Feature Relationships — CGPA vs Targets
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.boxplot(data=df, x='Graduation_Status', y='CGPA', ax=axes[0],
            palette=['#e74c3c', '#2ecc71'], width=0.4)
axes[0].set_title('CGPA by Graduation Status', fontsize=13, fontweight='bold')
axes[0].set_xticklabels(['Not Graduated', 'Graduated'])

sns.boxplot(data=df, x='Placement_Status', y='CGPA', ax=axes[1],
            palette=['#e67e22', '#3498db'], width=0.4)
axes[1].set_title('CGPA by Placement Status', fontsize=13, fontweight='bold')
axes[1].set_xticklabels(['Not Placed', 'Placed'])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4.9b Feature Relationships — Attendance, Aptitude, Backlogs, Projects
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.violinplot(data=df, x='Graduation_Status', y='Attendance_Percentage',
               ax=axes[0, 0], palette=['#e74c3c', '#2ecc71'], inner='quartile')
axes[0, 0].set_title('Attendance by Graduation Status', fontweight='bold')
axes[0, 0].set_xticklabels(['Not Graduated', 'Graduated'])

sns.violinplot(data=df, x='Placement_Status', y='Aptitude_Test_Score',
               ax=axes[0, 1], palette=['#e67e22', '#3498db'], inner='quartile')
axes[0, 1].set_title('Aptitude Score by Placement Status', fontweight='bold')
axes[0, 1].set_xticklabels(['Not Placed', 'Placed'])

sns.countplot(data=df, x='Backlogs', hue='Graduation_Status',
              ax=axes[1, 0], palette=['#e74c3c', '#2ecc71'])
axes[1, 0].set_title('Backlogs by Graduation Status', fontweight='bold')
axes[1, 0].legend(title='Graduated', labels=['No', 'Yes'])

sns.countplot(data=df, x='Projects_Completed', hue='Placement_Status',
              ax=axes[1, 1], palette=['#e67e22', '#3498db'])
axes[1, 1].set_title('Projects by Placement Status', fontweight='bold')
axes[1, 1].legend(title='Placed', labels=['No', 'Yes'])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4.9c Feature Relationships — Gender vs Targets
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pd.crosstab(df['Gender'], df['Graduation_Status'], normalize='index').plot(
    kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'], edgecolor='black'
)
axes[0].set_title('Graduation Rate by Gender', fontweight='bold')
axes[0].set_ylabel('Proportion')
axes[0].legend(['Not Graduated', 'Graduated'])
axes[0].tick_params(axis='x', rotation=0)

pd.crosstab(df['Gender'], df['Placement_Status'], normalize='index').plot(
    kind='bar', ax=axes[1], color=['#e67e22', '#3498db'], edgecolor='black'
)
axes[1].set_title('Placement Rate by Gender', fontweight='bold')
axes[1].set_ylabel('Proportion')
axes[1].legend(['Not Placed', 'Placed'])
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

---

## 5. Data Preprocessing

Data preprocessing transforms raw data into a clean, structured format suitable for machine learning models. This section covers:

1. **Missing value treatment** — Impute missing values with appropriate statistics.
2. **Outlier handling** — Cap extreme values using the IQR method.
3. **Encoding** — Convert categorical variables into numerical form.
4. **Feature scaling** — Standardize features to have zero mean and unit variance.
5. **Train-test split** — Divide data into training and testing sets.

### 5.1 Missing Value Treatment

**Strategy:** We impute missing numerical values with the **median** (robust to outliers) and missing categorical values with the **mode** (most frequent value).

In [ ]:
# ============================================================
# 5.1 Missing Value Treatment
# ============================================================

print("🔧 MISSING VALUE TREATMENT")
print("=" * 50)
print(f"Missing values BEFORE treatment: {df.isnull().sum().sum()}")

# Impute numerical columns with median
num_cols_with_missing = df.select_dtypes(include=[np.number]).columns[
    df.select_dtypes(include=[np.number]).isnull().any()
].tolist()

for col in num_cols_with_missing:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)
    print(f"  ✅ '{col}' — imputed with median: {median_val}")

# Impute categorical columns with mode
cat_cols_with_missing = df.select_dtypes(include=['object']).columns[
    df.select_dtypes(include=['object']).isnull().any()
].tolist()

for col in cat_cols_with_missing:
    mode_val = df[col].mode()[0]
    df[col].fillna(mode_val, inplace=True)
    print(f"  ✅ '{col}' — imputed with mode: {mode_val}")

print(f"\nMissing values AFTER treatment: {df.isnull().sum().sum()}")
print("✅ All missing values handled successfully!")

### 5.2 Outlier Handling

**Strategy:** We use the **Interquartile Range (IQR)** method to cap outliers. Values below `Q1 - 1.5 * IQR` are clipped to the lower bound, and values above `Q3 + 1.5 * IQR` are clipped to the upper bound. This method preserves the data distribution while limiting the influence of extreme values.

> **Why capping instead of removal?** Removing outliers reduces the dataset size and may discard genuinely informative data points. Capping (winsorizing) retains all records while bounding extreme values.

In [ ]:
# ============================================================
# 5.2 Outlier Handling (IQR Capping)
# ============================================================

# Columns to check for outliers (continuous numerical features only)
outlier_cols = [
    'Attendance_Percentage', 'CGPA', 'Internal_Marks', 'External_Marks',
    'Study_Hours_Per_Week', 'Communication_Skills_Score', 'Aptitude_Test_Score'
]

print("🔧 OUTLIER HANDLING (IQR Capping)")
print("=" * 50)

for col in outlier_cols:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        n_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()

        if n_outliers > 0:
            df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
            print(f"  ✅ '{col}' — {n_outliers} outliers capped "
                  f"[{lower_bound:.2f}, {upper_bound:.2f}]")
        else:
            print(f"  ℹ️  '{col}' — no outliers detected")

print("\n✅ Outlier treatment complete!")

### 5.3 Label Encoding / One-Hot Encoding

Machine learning models require numerical inputs. We convert categorical features as follows:

- **Binary categorical features** (e.g., Gender with 2 classes) → **Label Encoding** (0/1).
- **Multi-class categorical features** (if any) → **One-Hot Encoding** to avoid implying ordinal relationships.

> **Note:** Student_ID is a unique identifier and should be dropped before model training — it has no predictive value.

In [ ]:
# ============================================================
# 5.3 Encoding Categorical Variables
# ============================================================

print("🔧 ENCODING CATEGORICAL VARIABLES")
print("=" * 50)

# Drop Student_ID — it is a unique identifier, not a feature
if 'Student_ID' in df.columns:
    df = df.drop(columns=['Student_ID'])
    print("  ✅ 'Student_ID' column dropped (unique identifier, not predictive).")

# Identify categorical columns
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"  📋 Categorical columns found: {cat_cols}")

# Encode each categorical column
le_dict = {}  # Store label encoders for inverse transform later

for col in cat_cols:
    n_unique = df[col].nunique()
    if n_unique == 2:
        # Binary → Label Encoding
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        le_dict[col] = le
        print(f"  ✅ '{col}' — Label Encoded "
              f"(Binary: {list(le.classes_)} → [0, 1])")
    else:
        # Multi-class → One-Hot Encoding
        df = pd.get_dummies(df, columns=[col], drop_first=True, dtype=int)
        print(f"  ✅ '{col}' — One-Hot Encoded ({n_unique} categories)")

print(f"\n📐 Shape after encoding: {df.shape}")
print(f"\n📋 Final columns:")
print(df.columns.tolist())

### 5.4 Feature Scaling

**StandardScaler** standardizes features by removing the mean and scaling to unit variance: `z = (x - μ) / σ`.

This is important for:
- **Logistic Regression** — convergence speed and coefficient interpretation.
- **Distance-based algorithms** — ensuring all features contribute equally.
- **Gradient-based optimization** — helping gradients flow evenly across features.

> **Note:** Tree-based models (Decision Tree, Random Forest, XGBoost) are generally invariant to feature scaling, but we scale anyway for consistency.

In [ ]:
# ============================================================
# 5.4 Feature Scaling
# ============================================================

# Separate features and targets
target_cols = ['Graduation_Status', 'Placement_Status']
feature_cols = [c for c in df.columns if c not in target_cols]

print("🔧 FEATURE SCALING (StandardScaler)")
print("=" * 50)
print(f"  Features to scale: {len(feature_cols)} columns")

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[feature_cols] = scaler.fit_transform(df[feature_cols])

print("  ✅ Features scaled using StandardScaler (mean=0, std=1)")
print("\n📊 Sample of scaled data:")
df_scaled.head()

### 5.5 Train-Test Split

We split the dataset into **training (80%)** and **testing (20%)** sets. The training set is used to fit models, while the test set is held out for unbiased performance evaluation.

We use `stratify` to maintain the same class proportions in both splits, which is especially important for imbalanced datasets.

In [ ]:
# ============================================================
# 5.5 Train-Test Split
# ============================================================

# Features
X = df_scaled[feature_cols]

# Target 1: Graduation Status
y_grad = df_scaled['Graduation_Status']

# Target 2: Placement Status
y_place = df_scaled['Placement_Status']

# --- Split for Graduation Prediction ---
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X, y_grad, test_size=0.2, random_state=42, stratify=y_grad
)

# --- Split for Placement Prediction ---
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X, y_place, test_size=0.2, random_state=42, stratify=y_place
)

print("🔧 TRAIN-TEST SPLIT (80/20)")
print("=" * 50)
print(f"\n🎓 Graduation Prediction:")
print(f"   Training set : {X_train_g.shape[0]} samples")
print(f"   Testing set  : {X_test_g.shape[0]} samples")
print(f"   Class distribution (train): {dict(y_train_g.value_counts())}")
print(f"   Class distribution (test) : {dict(y_test_g.value_counts())}")

print(f"\n💼 Placement Prediction:")
print(f"   Training set : {X_train_p.shape[0]} samples")
print(f"   Testing set  : {X_test_p.shape[0]} samples")
print(f"   Class distribution (train): {dict(y_train_p.value_counts())}")
print(f"   Class distribution (test) : {dict(y_test_p.value_counts())}")

---

## 6. Feature Engineering

Feature engineering creates new informative features from existing ones to improve model performance. Well-crafted features can capture domain knowledge that raw features alone cannot express.

We engineer the following features:

| New Feature | Formula | Rationale |
|---|---|---|
| `Total_Marks` | Internal + External | Combined academic performance |
| `Marks_Ratio` | Internal / (Internal + External) | Balance between internal and external assessment |
| `Academic_Index` | 0.4×CGPA + 0.3×Attendance + 0.3×Total_Marks | Composite academic health score |
| `Employability_Score` | 0.3×Comm + 0.3×Aptitude + 0.2×Projects + 0.2×Internship | Composite job-readiness index |
| `Risk_Flag` | backlogs>2 AND attendance<60 | At-risk student indicator |

> **Important:** We perform feature engineering on the **original (unscaled)** DataFrame, then re-scale all features.

In [ ]:
# ============================================================
# 6. Feature Engineering
# ============================================================

print("🔧 FEATURE ENGINEERING")
print("=" * 50)

# Work on the original (unscaled) DataFrame
df_eng = df.copy()

# 1. Total Marks: Sum of internal and external marks
#    Rationale: Captures overall academic performance in a single metric.
df_eng['Total_Marks'] = df_eng['Internal_Marks'] + df_eng['External_Marks']
print("  ✅ 'Total_Marks' = Internal_Marks + External_Marks")

# 2. Marks Ratio: Proportion of internal marks to total
#    Rationale: Students who score well internally but poorly externally
#    (or vice versa) may have different graduation/placement patterns.
df_eng['Marks_Ratio'] = (
    df_eng['Internal_Marks'] / (df_eng['Total_Marks'] + 1e-8)
)
print("  ✅ 'Marks_Ratio' = Internal_Marks / Total_Marks")

# 3. Academic Index: Weighted composite of core academic metrics
#    Rationale: Provides a holistic view of academic health,
#    combining CGPA, attendance, and marks into a single score.
df_eng['Academic_Index'] = (
    0.4 * (df_eng['CGPA'] / 10) * 100
    + 0.3 * df_eng['Attendance_Percentage']
    + 0.3 * (df_eng['Total_Marks'] / 200) * 100
)
print("  ✅ 'Academic_Index' = 0.4×CGPA_norm + 0.3×Attendance + 0.3×Marks_norm")

# 4. Employability Score: Weighted composite of job-readiness factors
#    Rationale: Combines soft skills, technical skills, and experience
#    into a single metric that indicates placement readiness.
df_eng['Employability_Score'] = (
    0.30 * (df_eng['Communication_Skills_Score'] / 10) * 100
    + 0.30 * df_eng['Aptitude_Test_Score']
    + 0.20 * (df_eng['Projects_Completed'] / 5) * 100
    + 0.20 * df_eng['Internship_Experience'] * 100
)
print("  ✅ 'Employability_Score' = 0.3×Comm + 0.3×Aptitude "
      "+ 0.2×Projects + 0.2×Internship")

# 5. Risk Flag: Binary flag for at-risk students
#    Rationale: Students with many backlogs AND low attendance are at
#    high risk of not graduating. This captures an interaction effect.
df_eng['Risk_Flag'] = (
    (df_eng['Backlogs'] > 2) & (df_eng['Attendance_Percentage'] < 60)
).astype(int)
print("  ✅ 'Risk_Flag' = 1 if Backlogs > 2 AND Attendance < 60, else 0")

print(f"\n📐 Shape after feature engineering: {df_eng.shape}")
print(f"   New features added: 5")

In [ ]:
# --- Re-do preprocessing with engineered features ---

# Update feature columns
target_cols = ['Graduation_Status', 'Placement_Status']
feature_cols_eng = [c for c in df_eng.columns if c not in target_cols]

# Re-scale
scaler_eng = StandardScaler()
df_eng_scaled = df_eng.copy()
df_eng_scaled[feature_cols_eng] = scaler_eng.fit_transform(
    df_eng[feature_cols_eng]
)

# Re-split
X_eng = df_eng_scaled[feature_cols_eng]
y_grad = df_eng_scaled['Graduation_Status']
y_place = df_eng_scaled['Placement_Status']

X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_eng, y_grad, test_size=0.2, random_state=42, stratify=y_grad
)
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_eng, y_place, test_size=0.2, random_state=42, stratify=y_place
)

print("✅ Preprocessing re-applied with engineered features.")
print(f"   Training features: {X_train_g.shape}")
print(f"   Testing features : {X_test_g.shape}")

---

## 7. Graduation Prediction Model

In this section, we train multiple classification models to predict **Graduation Status** (binary: 0 = Not Graduated, 1 = Graduated). We then evaluate and compare all models using standard classification metrics.

### Models Trained:
1. **Logistic Regression** — Simple, interpretable linear classifier.
2. **Decision Tree** — Non-linear, interpretable, but prone to overfitting.
3. **Random Forest** — Ensemble of decision trees; reduces overfitting.
4. **Gradient Boosting** — Sequential ensemble method; often top-performing.
5. **XGBoost** — Optimized gradient boosting; industry standard (if available).

### Evaluation Metrics:
- **Accuracy** — Overall correctness.
- **Precision** — Of predicted positives, how many are actually positive?
- **Recall** — Of actual positives, how many did we catch?
- **F1-Score** — Harmonic mean of precision and recall.
- **ROC-AUC** — Area under the ROC curve; measures discrimination ability.

In [ ]:
# ============================================================
# 7. Helper Function — Train & Evaluate Multiple Models
# ============================================================

def train_evaluate_models(X_train, X_test, y_train, y_test, task_name="Task"):
    """
    Train multiple classification models and evaluate them.

    Parameters
    ----------
    X_train, X_test : pd.DataFrame
        Training and testing feature matrices.
    y_train, y_test : pd.Series
        Training and testing target vectors.
    task_name : str
        Name of the prediction task (for display purposes).

    Returns
    -------
    results_df : pd.DataFrame
        Comparison table of model performance.
    trained_models : dict
        Dictionary of trained model objects.
    """

    # --- Define models ---
    models = {
        'Logistic Regression': LogisticRegression(
            max_iter=1000, random_state=42, solver='lbfgs'
        ),
        'Decision Tree': DecisionTreeClassifier(
            random_state=42, max_depth=10
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=200, random_state=42, n_jobs=-1
        ),
        'Gradient Boosting': GradientBoostingClassifier(
            n_estimators=200, random_state=42, learning_rate=0.1
        ),
    }

    # Add XGBoost if available
    if XGBOOST_AVAILABLE:
        models['XGBoost'] = XGBClassifier(
            n_estimators=200, random_state=42, learning_rate=0.1,
            use_label_encoder=False, eval_metric='logloss', verbosity=0
        )

    # --- Train and evaluate each model ---
    results = []
    trained_models = {}

    print(f"\n{'='*70}")
    print(f"  🏋️ TRAINING MODELS FOR: {task_name.upper()}")
    print(f"{'='*70}")

    for name, model in models.items():
        print(f"\n  🔄 Training {name}...", end=" ")

        # Train
        model.fit(X_train, y_train)
        trained_models[name] = model

        # Predict
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        # Evaluate
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc = roc_auc_score(y_test, y_prob)

        results.append({
            'Model': name,
            'Accuracy': round(acc, 4),
            'Precision': round(prec, 4),
            'Recall': round(rec, 4),
            'F1-Score': round(f1, 4),
            'ROC-AUC': round(roc, 4),
        })

        print(f"✅ Accuracy: {acc:.4f} | F1: {f1:.4f} | ROC-AUC: {roc:.4f}")

    # --- Create results DataFrame ---
    results_df = pd.DataFrame(results).set_index('Model')
    results_df = results_df.sort_values(by='ROC-AUC', ascending=False)

    return results_df, trained_models

In [ ]:
# ============================================================
# 7. Train Models for Graduation Prediction
# ============================================================

grad_results, grad_models = train_evaluate_models(
    X_train_g, X_test_g, y_train_g, y_test_g,
    task_name="Graduation Prediction"
)

print("\n" + "=" * 70)
print("📊 GRADUATION PREDICTION — MODEL COMPARISON TABLE")
print("=" * 70)
grad_results.style.background_gradient(cmap='Greens', axis=0)

In [ ]:
# ============================================================
# 7. Graduation Prediction — Confusion Matrices
# ============================================================

n_models = len(grad_models)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
if n_models == 1:
    axes = [axes]

for idx, (name, model) in enumerate(grad_models.items()):
    y_pred = model.predict(X_test_g)
    cm = confusion_matrix(y_test_g, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['Not Grad', 'Graduated']).plot(
        ax=axes[idx], cmap='Greens', colorbar=False
    )
    axes[idx].set_title(f'{name}', fontsize=11, fontweight='bold')

fig.suptitle('🎓 Graduation Prediction — Confusion Matrices',
             fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

---

## 8. Placement Prediction Model

We now repeat the modeling process for the **Placement Status** target variable (binary: 0 = Not Placed, 1 = Placed). The same set of models and evaluation metrics are used for a fair comparison.

In [ ]:
# ============================================================
# 8. Train Models for Placement Prediction
# ============================================================

place_results, place_models = train_evaluate_models(
    X_train_p, X_test_p, y_train_p, y_test_p,
    task_name="Placement Prediction"
)

print("\n" + "=" * 70)
print("📊 PLACEMENT PREDICTION — MODEL COMPARISON TABLE")
print("=" * 70)
place_results.style.background_gradient(cmap='Blues', axis=0)

In [ ]:
# ============================================================
# 8. Placement Prediction — Confusion Matrices
# ============================================================

n_models = len(place_models)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
if n_models == 1:
    axes = [axes]

for idx, (name, model) in enumerate(place_models.items()):
    y_pred = model.predict(X_test_p)
    cm = confusion_matrix(y_test_p, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['Not Placed', 'Placed']).plot(
        ax=axes[idx], cmap='Blues', colorbar=False
    )
    axes[idx].set_title(f'{name}', fontsize=11, fontweight='bold')

fig.suptitle('💼 Placement Prediction — Confusion Matrices',
             fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

---

## 9. Hyperparameter Tuning

Hyperparameter tuning optimizes model performance by searching for the best combination of model parameters. We use **RandomizedSearchCV** for efficiency — it samples a random subset of the parameter space rather than exhaustively searching all combinations (as GridSearchCV would).

We tune the **best-performing model** from each task based on the ROC-AUC score.

> **Why RandomizedSearchCV over GridSearchCV?** For high-dimensional parameter spaces, RandomizedSearchCV is significantly faster while still finding near-optimal configurations.

In [ ]:
# ============================================================
# 9. Hyperparameter Tuning — Helper Function
# ============================================================

def tune_best_model(results_df, models_dict, X_train, y_train,
                    X_test, y_test, task_name):
    """
    Tune the best-performing model using RandomizedSearchCV.

    Parameters
    ----------
    results_df : pd.DataFrame
        Model comparison table (sorted by ROC-AUC).
    models_dict : dict
        Dictionary of trained model objects.
    X_train, y_train, X_test, y_test : array-like
        Training and testing data.
    task_name : str
        Name of the prediction task.

    Returns
    -------
    best_tuned_model : estimator
        The tuned model.
    best_model_name : str
        Name of the best model.
    """

    best_model_name = results_df.index[0]  # Top model by ROC-AUC
    print(f"\n🏆 Best model for {task_name}: {best_model_name}")
    print(f"   (Based on ROC-AUC = {results_df.iloc[0]['ROC-AUC']})")

    # --- Define parameter grids for each model type ---
    param_grids = {
        'Logistic Regression': {
            'C': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
            'penalty': ['l2'],
            'solver': ['lbfgs', 'liblinear'],
        },
        'Decision Tree': {
            'max_depth': [3, 5, 7, 10, 15, None],
            'min_samples_split': [2, 5, 10, 20],
            'min_samples_leaf': [1, 2, 5, 10],
            'criterion': ['gini', 'entropy'],
        },
        'Random Forest': {
            'n_estimators': [100, 200, 300, 500],
            'max_depth': [5, 10, 15, 20, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 5],
            'max_features': ['sqrt', 'log2'],
        },
        'Gradient Boosting': {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'max_depth': [3, 5, 7],
            'subsample': [0.7, 0.8, 0.9, 1.0],
            'min_samples_split': [2, 5, 10],
        },
        'XGBoost': {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'max_depth': [3, 5, 7],
            'subsample': [0.7, 0.8, 0.9, 1.0],
            'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
            'reg_alpha': [0, 0.1, 1],
            'reg_lambda': [0.1, 1, 10],
        },
    }

    if best_model_name not in param_grids:
        print(f"   ⚠️ No tuning grid defined for {best_model_name}. "
              f"Skipping.")
        return models_dict[best_model_name], best_model_name

    param_grid = param_grids[best_model_name]
    base_model = models_dict[best_model_name]

    print(f"\n🔍 Running RandomizedSearchCV with 50 iterations, "
          f"5-fold CV...")

    search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=param_grid,
        n_iter=50,
        cv=5,
        scoring='roc_auc',
        random_state=42,
        n_jobs=-1,
        verbose=0,
    )

    search.fit(X_train, y_train)

    print(f"\n✅ Tuning Complete!")
    print(f"   Best Parameters: {search.best_params_}")
    print(f"   Best CV ROC-AUC: {search.best_score_:.4f}")

    # --- Evaluate tuned model on test set ---
    best_tuned_model = search.best_estimator_
    y_pred = best_tuned_model.predict(X_test)
    y_prob = best_tuned_model.predict_proba(X_test)[:, 1]

    print(f"\n📊 Tuned Model — Test Set Performance:")
    print(f"   Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
    print(f"   Precision : {precision_score(y_test, y_pred):.4f}")
    print(f"   Recall    : {recall_score(y_test, y_pred):.4f}")
    print(f"   F1-Score  : {f1_score(y_test, y_pred):.4f}")
    print(f"   ROC-AUC   : {roc_auc_score(y_test, y_prob):.4f}")

    return best_tuned_model, best_model_name

In [ ]:
# --- Tune best Graduation model ---
print("🎓 HYPERPARAMETER TUNING — GRADUATION PREDICTION")
print("=" * 60)

best_grad_model, best_grad_name = tune_best_model(
    grad_results, grad_models,
    X_train_g, y_train_g, X_test_g, y_test_g,
    task_name="Graduation Prediction"
)

In [ ]:
# --- Tune best Placement model ---
print("\n💼 HYPERPARAMETER TUNING — PLACEMENT PREDICTION")
print("=" * 60)

best_place_model, best_place_name = tune_best_model(
    place_results, place_models,
    X_train_p, y_train_p, X_test_p, y_test_p,
    task_name="Placement Prediction"
)

---

## 10. Feature Importance

Feature importance analysis reveals which student attributes have the most influence on graduation and placement predictions. This is valuable for:

- **Interpretability:** Understanding what drives model predictions.
- **Actionable Insights:** Identifying areas where institutions can intervene.
- **Feature Selection:** Potentially simplifying models by removing low-importance features.

We extract feature importances from **Random Forest** and **XGBoost** (if available), as tree-based models provide built-in feature importance measures.

In [ ]:
# ============================================================
# 10. Feature Importance — Helper Function
# ============================================================

def plot_feature_importance(model, feature_names, title, color, top_n=15):
    """
    Plot feature importances from a tree-based model.

    Parameters
    ----------
    model : estimator
        Trained tree-based model with feature_importances_ attribute.
    feature_names : list
        List of feature names.
    title : str
        Plot title.
    color : str
        Bar color.
    top_n : int
        Number of top features to display.
    """
    importances = pd.Series(
        model.feature_importances_, index=feature_names
    ).sort_values(ascending=True).tail(top_n)

    fig, ax = plt.subplots(figsize=(10, 7))
    importances.plot(kind='barh', color=color, edgecolor='black', ax=ax)
    ax.set_xlabel('Feature Importance', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.bar_label(ax.containers[0], fmt='%.3f', padding=3, fontsize=9)
    plt.tight_layout()
    plt.show()

    return importances

In [ ]:
# --- Feature Importance: Graduation ---
print("🎓 FEATURE IMPORTANCE — GRADUATION PREDICTION")
print("=" * 60)

if 'Random Forest' in grad_models:
    print("\n🌲 Random Forest Feature Importance:")
    grad_rf_imp = plot_feature_importance(
        grad_models['Random Forest'], feature_cols_eng,
        '🎓 Random Forest — Graduation Feature Importance',
        color='#2ecc71'
    )

if XGBOOST_AVAILABLE and 'XGBoost' in grad_models:
    print("\n🚀 XGBoost Feature Importance:")
    grad_xgb_imp = plot_feature_importance(
        grad_models['XGBoost'], feature_cols_eng,
        '🎓 XGBoost — Graduation Feature Importance',
        color='#27ae60'
    )

In [ ]:
# --- Feature Importance: Placement ---
print("💼 FEATURE IMPORTANCE — PLACEMENT PREDICTION")
print("=" * 60)

if 'Random Forest' in place_models:
    print("\n🌲 Random Forest Feature Importance:")
    place_rf_imp = plot_feature_importance(
        place_models['Random Forest'], feature_cols_eng,
        '💼 Random Forest — Placement Feature Importance',
        color='#3498db'
    )

if XGBOOST_AVAILABLE and 'XGBoost' in place_models:
    print("\n🚀 XGBoost Feature Importance:")
    place_xgb_imp = plot_feature_importance(
        place_models['XGBoost'], feature_cols_eng,
        '💼 XGBoost — Placement Feature Importance',
        color='#2980b9'
    )

### Feature Importance Insights

**Key observations:**

- **Graduation:** The most important features for predicting graduation are typically CGPA, Attendance Percentage, Academic Index, and Backlogs. This makes intuitive sense — academic engagement and performance are the strongest indicators of successful degree completion.

- **Placement:** For placement prediction, Employability Score, Communication Skills, Aptitude Test Score, and Internship Experience tend to dominate. This highlights that employers value a combination of technical aptitude and soft skills.

- **Engineered Features:** The composite features (Academic_Index, Employability_Score) often rank among the top predictors, validating our feature engineering approach.

---

## 11. Model Comparison

In this section, we consolidate results from both prediction tasks and create comprehensive visualizations to compare model performance. This helps us make an informed decision about the **final model selection**.

In [ ]:
# ============================================================
# 11. Model Comparison — Summary Tables
# ============================================================

print("📊 FINAL MODEL COMPARISON")
print("=" * 70)

print("\n🎓 Graduation Prediction — All Models:")
print(grad_results.to_string())

print("\n💼 Placement Prediction — All Models:")
print(place_results.to_string())

In [ ]:
# --- Bar Chart: Model Accuracy Comparison ---

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Graduation
grad_results['Accuracy'].sort_values().plot(
    kind='barh', ax=axes[0], color='#2ecc71', edgecolor='black', alpha=0.85
)
axes[0].set_title('🎓 Graduation — Model Accuracy',
                  fontsize=14, fontweight='bold')
axes[0].set_xlabel('Accuracy', fontsize=12)
axes[0].bar_label(axes[0].containers[0], fmt='%.4f', padding=3)
axes[0].set_xlim(0, 1.05)

# Placement
place_results['Accuracy'].sort_values().plot(
    kind='barh', ax=axes[1], color='#3498db', edgecolor='black', alpha=0.85
)
axes[1].set_title('💼 Placement — Model Accuracy',
                  fontsize=14, fontweight='bold')
axes[1].set_xlabel('Accuracy', fontsize=12)
axes[1].bar_label(axes[1].containers[0], fmt='%.4f', padding=3)
axes[1].set_xlim(0, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# --- Grouped Bar Chart: All Metrics Comparison ---

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Graduation
grad_results.plot(
    kind='bar', ax=axes[0], colormap='Greens', edgecolor='black', alpha=0.85
)
axes[0].set_title('🎓 Graduation Prediction — All Metrics by Model',
                  fontsize=14, fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.1)
axes[0].legend(loc='lower right', fontsize=9)
axes[0].tick_params(axis='x', rotation=15)

# Placement
place_results.plot(
    kind='bar', ax=axes[1], colormap='Blues', edgecolor='black', alpha=0.85
)
axes[1].set_title('💼 Placement Prediction — All Metrics by Model',
                  fontsize=14, fontweight='bold')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0, 1.1)
axes[1].legend(loc='lower right', fontsize=9)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# --- ROC Curve Comparison ---

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Graduation ROC Curves
for name, model in grad_models.items():
    RocCurveDisplay.from_estimator(
        model, X_test_g, y_test_g, ax=axes[0], name=name, alpha=0.8
    )
axes[0].set_title('🎓 Graduation — ROC Curves',
                  fontsize=14, fontweight='bold')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.5)')
axes[0].legend(fontsize=9)

# Placement ROC Curves
for name, model in place_models.items():
    RocCurveDisplay.from_estimator(
        model, X_test_p, y_test_p, ax=axes[1], name=name, alpha=0.8
    )
axes[1].set_title('💼 Placement — ROC Curves',
                  fontsize=14, fontweight='bold')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.5)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

### 11.1 Final Model Selection Justification

Based on the comprehensive evaluation above, we select the final models using the following criteria:

1. **ROC-AUC** as the primary metric — it measures the model's ability to discriminate between classes across all thresholds.
2. **F1-Score** as a secondary metric — it balances precision and recall, which is important when both false positives and false negatives carry costs.
3. **Interpretability** — in educational settings, stakeholders need to understand *why* a prediction was made.

In [ ]:
# ============================================================
# 11.1 Final Model Selection
# ============================================================

print("🏆 FINAL MODEL SELECTION")
print("=" * 60)

print(f"\n🎓 Graduation Prediction:")
print(f"   Selected Model : {best_grad_name} (tuned)")
print(f"   Rationale      : Highest ROC-AUC among all models after tuning.")

print(f"\n💼 Placement Prediction:")
print(f"   Selected Model : {best_place_name} (tuned)")
print(f"   Rationale      : Highest ROC-AUC among all models after tuning.")

---

## 12. Predictions on New Student Data

In this section, we demonstrate how the trained models can be used to predict graduation and placement outcomes for **new, unseen students**. This simulates a real-world deployment scenario where the model receives fresh student data and returns predictions along with confidence probabilities.

In [ ]:
# ============================================================
# 12. Predictions on New Student Data
# ============================================================

# --- Create sample new student data ---
# These are hypothetical students with varying profiles.

new_students_raw = pd.DataFrame({
    'Gender': [1, 0, 1, 0, 1],             # 1=Male, 0=Female
    'Age': [21, 23, 20, 22, 25],
    'Attendance_Percentage': [92.0, 55.0, 78.0, 88.0, 40.0],
    'CGPA': [8.7, 5.2, 7.1, 9.0, 4.0],
    'Internal_Marks': [82.0, 35.0, 60.0, 90.0, 25.0],
    'External_Marks': [78.0, 40.0, 55.0, 85.0, 20.0],
    'Backlogs': [0, 4, 1, 0, 6],
    'Study_Hours_Per_Week': [25.0, 5.0, 15.0, 30.0, 3.0],
    'Internship_Experience': [1, 0, 1, 1, 0],
    'Projects_Completed': [4, 0, 2, 5, 0],
    'Communication_Skills_Score': [8.5, 3.0, 6.5, 9.0, 2.0],
    'Aptitude_Test_Score': [85.0, 30.0, 60.0, 92.0, 25.0],
}, index=['Student_A', 'Student_B', 'Student_C', 'Student_D', 'Student_E'])

print("📝 NEW STUDENT DATA")
print("=" * 60)
new_students_raw

In [ ]:
# --- Apply feature engineering to new data ---
new_students = new_students_raw.copy()

new_students['Total_Marks'] = (
    new_students['Internal_Marks'] + new_students['External_Marks']
)
new_students['Marks_Ratio'] = (
    new_students['Internal_Marks'] / (new_students['Total_Marks'] + 1e-8)
)
new_students['Academic_Index'] = (
    0.4 * (new_students['CGPA'] / 10) * 100
    + 0.3 * new_students['Attendance_Percentage']
    + 0.3 * (new_students['Total_Marks'] / 200) * 100
)
new_students['Employability_Score'] = (
    0.30 * (new_students['Communication_Skills_Score'] / 10) * 100
    + 0.30 * new_students['Aptitude_Test_Score']
    + 0.20 * (new_students['Projects_Completed'] / 5) * 100
    + 0.20 * new_students['Internship_Experience'] * 100
)
new_students['Risk_Flag'] = (
    (new_students['Backlogs'] > 2)
    & (new_students['Attendance_Percentage'] < 60)
).astype(int)

# --- Ensure column order matches training data ---
new_students = new_students[feature_cols_eng]

# --- Scale features ---
new_students_scaled = pd.DataFrame(
    scaler_eng.transform(new_students),
    columns=feature_cols_eng,
    index=new_students.index
)

print("✅ New student data preprocessed and scaled.")
print(f"   Shape: {new_students_scaled.shape}")

In [ ]:
# --- Generate Predictions ---

# Graduation predictions
grad_pred = best_grad_model.predict(new_students_scaled)
grad_prob = best_grad_model.predict_proba(new_students_scaled)

# Placement predictions
place_pred = best_place_model.predict(new_students_scaled)
place_prob = best_place_model.predict_proba(new_students_scaled)

# --- Assemble results ---
predictions_df = pd.DataFrame({
    'Student': new_students_raw.index,
    'CGPA': new_students_raw['CGPA'].values,
    'Attendance': new_students_raw['Attendance_Percentage'].values,
    'Backlogs': new_students_raw['Backlogs'].values,
    'Graduation_Prediction': [
        '✅ Graduate' if p == 1 else '❌ At Risk' for p in grad_pred
    ],
    'Graduation_Probability': [
        f"{p[1]*100:.1f}%" for p in grad_prob
    ],
    'Placement_Prediction': [
        '✅ Placed' if p == 1 else '❌ Not Placed' for p in place_pred
    ],
    'Placement_Probability': [
        f"{p[1]*100:.1f}%" for p in place_prob
    ],
}).set_index('Student')

print("\n🔮 PREDICTION RESULTS")
print("=" * 80)
predictions_df

In [ ]:
# --- Visualize prediction probabilities ---

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

students = new_students_raw.index.tolist()
x = np.arange(len(students))
width = 0.35

# Graduation Probabilities
grad_probs_vals = [p[1] for p in grad_prob]
bars1 = axes[0].bar(x, grad_probs_vals, width,
                    color='#2ecc71', edgecolor='black', alpha=0.85)
axes[0].axhline(y=0.5, color='red', linestyle='--', linewidth=1.5,
                label='Threshold (0.5)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(students, rotation=0)
axes[0].set_ylabel('Probability')
axes[0].set_title('🎓 Graduation Probability',
                  fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 1.1)
axes[0].bar_label(bars1, fmt='%.2f', padding=3)
axes[0].legend()

# Placement Probabilities
place_probs_vals = [p[1] for p in place_prob]
bars2 = axes[1].bar(x, place_probs_vals, width,
                    color='#3498db', edgecolor='black', alpha=0.85)
axes[1].axhline(y=0.5, color='red', linestyle='--', linewidth=1.5,
                label='Threshold (0.5)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(students, rotation=0)
axes[1].set_ylabel('Probability')
axes[1].set_title('💼 Placement Probability',
                  fontsize=14, fontweight='bold')
axes[1].set_ylim(0, 1.1)
axes[1].bar_label(bars2, fmt='%.2f', padding=3)
axes[1].legend()

plt.tight_layout()
plt.show()

### 12.1 Interpretation of Predictions

| Student | Profile | Graduation | Placement | Interpretation |
|---|---|---|---|---|
| **Student_A** | High CGPA (8.7), good attendance, internship, strong skills | ✅ Graduate | ✅ Placed | Model-ideal student — strong in academics and employability. |
| **Student_B** | Low CGPA (5.2), low attendance, 4 backlogs, no internship | ❌ At Risk | ❌ Not Placed | Clear at-risk profile — needs immediate intervention. |
| **Student_C** | Average CGPA (7.1), decent attendance, has internship | ✅ Graduate | Depends | Borderline — small improvements could help placement. |
| **Student_D** | Excellent all-around (CGPA 9.0, 5 projects, strong skills) | ✅ Graduate | ✅ Placed | Top performer — high confidence in both outcomes. |
| **Student_E** | Very low metrics across the board, 6 backlogs | ❌ At Risk | ❌ Not Placed | Critical case — requires immediate academic support. |

---

## 13. Conclusion

### 13.1 Key Findings

1. **Multiple ML models** were successfully trained to predict both graduation and placement outcomes with strong performance metrics.

2. **Ensemble methods** (Random Forest, Gradient Boosting, XGBoost) consistently outperformed simpler models (Logistic Regression, Decision Tree) in terms of ROC-AUC and F1-score.

3. **Hyperparameter tuning** provided incremental but meaningful improvements in model performance.

4. **Feature engineering** (Academic Index, Employability Score, Risk Flag) added significant predictive value, with engineered features ranking among the top importance scores.

### 13.2 Most Important Factors Affecting Graduation

| Rank | Factor | Impact |
|------|--------|--------|
| 1 | **CGPA** | Strongest predictor — higher CGPA strongly correlates with graduation |
| 2 | **Attendance Percentage** | Regular attendance is critical for academic success |
| 3 | **Academic Index** (engineered) | Composite score captures overall academic engagement |
| 4 | **Backlogs** | Multiple backlogs are a strong negative indicator |
| 5 | **Study Hours Per Week** | Consistent study habits support timely graduation |

### 13.3 Most Important Factors Affecting Placement

| Rank | Factor | Impact |
|------|--------|--------|
| 1 | **Employability Score** (engineered) | Composite of skills and experience |
| 2 | **Aptitude Test Score** | Technical aptitude is crucial for placement |
| 3 | **Communication Skills Score** | Soft skills differentiate candidates |
| 4 | **CGPA** | Academic performance still matters for hiring |
| 5 | **Internship Experience** | Practical experience significantly boosts placement chances |

### 13.4 Recommendations for Educational Institutions

1. **Deploy an Early Warning System:** Use the graduation prediction model to flag at-risk students at the start of each semester. Students with predicted graduation probability below 50% should receive proactive counseling and academic support.

2. **Mandatory Attendance Monitoring:** Since attendance is a top predictor of graduation, institutions should implement strict attendance tracking with automated alerts for students falling below 70%.

3. **Focus on Employability Skills:** The placement model clearly shows that aptitude, communication skills, and internship experience are critical. Institutions should:
   - Integrate aptitude and soft-skills training into the curriculum.
   - Make internships mandatory (or strongly encouraged) by the 3rd year.
   - Provide mock interview and communication workshops.

4. **Backlog Remediation Programs:** Students with 2 or more backlogs are significantly more likely to not graduate. Fast-track remedial courses and summer backlog clearance programs can help.

5. **Project-Based Learning:** Encourage students to complete at least 2–3 projects, as project experience positively influences both graduation confidence and placement outcomes.

6. **Periodic Model Retraining:** As student demographics and market conditions evolve, retrain models periodically (e.g., annually) with fresh data to maintain prediction accuracy.

---

### 🏁 End of Notebook

This notebook provides a complete, end-to-end machine learning pipeline for predicting student graduation and placement outcomes. The dataset is loaded from `data/student_data.csv`, which can be regenerated using `data/generate_dataset.py` or replaced with a real-world dataset.

---

*Notebook prepared as part of the internship project: "Predict Success: Build ML Models for Graduation & Placement Forecasting"*